# Parte 3 — Recommendations Engine

Engine que recebe os dados de uma campanha e gera sugestões de melhoria ordenadas por impacto estimado.

**Fluxo:**
1. Carrega o modelo XGBoost treinado na Parte 2
2. Tenta gerar sugestões via DiCE (counterfactuals)
3. Se DiCE falhar, testa cada feature manualmente
4. Enriquece cada sugestão com estatísticas reais do dataset (plataforma × objetivo quando há amostras suficientes, senão só plataforma)


In [33]:
import joblib
import pandas as pd
import numpy as np
import dice_ml
from dice_ml import Dice

model               = joblib.load('models/xgb_klike.pkl')
feature_cols        = joblib.load('models/feature_columns.pkl')
mediana             = joblib.load('models/mediana_imputacao.pkl')
permitted_range     = joblib.load('models/permitted_range.pkl')
df_train            = pd.read_csv('models/df_dice_train.csv')

print(f'Modelo: {type(model).__name__}')
print(f'Features: {len(feature_cols)}')
print(f'Campanhas no treino: {len(df_train)}')


Modelo: XGBRegressor
Features: 94
Campanhas no treino: 350


## Pré-computa estatísticas do dataset

In [34]:
MIN_SAMPLES = 5
BINARY_FEATS = ['has_hook', 'has_face', 'has_subtitle', 'has_cta']
FEAT_STATS   = BINARY_FEATS + ['text_density_cat']

def _reconstruir_objective(df):
    """Reconstroi coluna 'objective' a partir das colunas OHE."""
    objectives = ['awareness', 'traffic', 'conversions', 'engagement', 'app_install']
    platforms  = ['Meta', 'TikTok', 'LinkedIn']
    result = pd.Series('unknown', index=df.index)
    for obj in objectives:
        for plat in platforms:
            col = f'objective_platform_{obj}_{plat}'
            if col in df.columns:
                result[df[col] == 1] = obj
    return result

def compute_stats(df, min_samples=MIN_SAMPLES):
    """
    Para cada plataforma x feature, calcula medias de CTR e engajamento:
    - com objetivo: se grupo tiver >= min_samples amostras
    - sem objetivo: fallback sempre disponivel
    """
    platforms = ['TikTok', 'Meta', 'LinkedIn']
    stats = {}

    # reconstroi objetivo antes de agrupar
    df = df.copy()
    df['objective'] = _reconstruir_objective(df)

    for plat in platforms:
        df_plat = df[df[f'platform_{plat}'] == 1].copy()
        stats[plat] = {}

        for feat in FEAT_STATS:
            # com objetivo — filtra grupos com amostras suficientes
            counts_obj = df_plat.groupby(['objective', feat]).size()
            means_obj  = df_plat.groupby(['objective', feat])[
                ['ctr_lg_mm', 'engagement_rate']
            ].mean()
            valid = {k for k, n in counts_obj.items() if n >= min_samples}
            obj_dict = {
                k: v for k, v in means_obj.to_dict(orient='index').items()
                if k in valid
            }

            # sem objetivo (fallback)
            plat_dict = df_plat.groupby(feat)[
                ['ctr_lg_mm', 'engagement_rate']
            ].mean().to_dict(orient='index')

            stats[plat][feat] = {'obj': obj_dict, 'plat': plat_dict}

    return stats

STATS = compute_stats(df_train)
print('Estatísticas calculadas.')


Estatísticas calculadas.


## Setup DiCE

In [35]:
FEATURES_RECOMENDAVEIS = [
    'has_hook', 'has_cta', 'has_face', 'has_subtitle',
    'text_density_cat', 'music_voice_ratio'
]

df_dice = df_train.copy()
d = dice_ml.Data(
    dataframe=df_dice,
    continuous_features=feature_cols,
    outcome_name='klike_score'
)
m   = dice_ml.Model(model=model, backend='sklearn', model_type='regressor')
exp = Dice(d, m, method='random')
print('DiCE pronto.')


DiCE pronto.


## Engine

In [36]:
TEXT_DENSITY_MAP = {0: 'low', 1: 'medium', 2: 'high'}

OBJECTIVE_LABEL = {
    'conversions': 'conversoes',
    'awareness':   'awareness',
    'traffic':     'trafego',
    'engagement':  'engajamento',
    'app_install': 'instalacao de app',
}

FEAT_LABEL = {
    'has_hook':          'Adicionar um gancho nos primeiros 3 segundos',
    'has_cta':           "Adicionar uma chamada para acao (ex: 'Saiba mais', 'Compre agora')",
    'has_face':          'Incluir rosto humano no video',
    'has_subtitle':      'Adicionar legendas',
    'text_density_cat':  'Reduzir a quantidade de texto na tela',
    'music_voice_ratio': 'Ajustar o equilibrio entre musica e voz',
}

FEAT_CONTEXTO_LABEL = {
    'has_hook':     'gancho nos primeiros segundos',
    'has_cta':      'chamada para acao',
    'has_face':     'rosto humano',
    'has_subtitle': 'legendas',
}

# ── Helpers ──────────────────────────────────────────────────────────────

def _get_platform(row):
    for p in ['TikTok', 'Meta', 'LinkedIn']:
        if row.get(f'platform_{p}', 0) == 1:
            return p
    return 'desconhecida'

def _get_objective(row):
    objectives = ['awareness', 'traffic', 'conversions', 'engagement', 'app_install']
    platforms  = ['Meta', 'TikTok', 'LinkedIn']
    for obj in objectives:
        for plat in platforms:
            if row.get(f'objective_platform_{obj}_{plat}', 0) == 1:
                return obj
    return 'unknown'

def _lookup(platform, feat, objective, val_com, val_sem):
    """Retorna (stats_com, stats_sem, usou_objetivo)."""
    s = STATS.get(platform, {}).get(feat, {})

    com = s['obj'].get((objective, float(val_com)),
          s['obj'].get((objective, int(val_com)), None))
    sem = s['obj'].get((objective, float(val_sem)),
          s['obj'].get((objective, int(val_sem)), None))

    if com and sem:
        return com, sem, True

    # fallback: so plataforma
    com = s['plat'].get(float(val_com), s['plat'].get(val_com, {}))
    sem = s['plat'].get(float(val_sem), s['plat'].get(val_sem, {}))
    return com, sem, False

def _contexto_binaria(platform, feat, objective):
    label  = FEAT_CONTEXTO_LABEL.get(feat, feat.replace('has_', ''))
    com, sem, usou_obj = _lookup(platform, feat, objective, 1, 0)

    if not com or not sem:
        return ''

    ctr_com = com.get('ctr_lg_mm', 0)
    ctr_sem = sem.get('ctr_lg_mm', 0)
    eng_com = com.get('engagement_rate', 0)
    eng_sem = sem.get('engagement_rate', 0)

    if usou_obj:
        obj_label = OBJECTIVE_LABEL.get(objective, objective)
        prefix = f'Para campanhas de {obj_label} no {platform}'
    else:
        prefix = f'No {platform}'

    return (
        f'{prefix}, videos COM {label} tem '
        f'taxa de cliques {(ctr_com-ctr_sem)/max(ctr_sem,1e-9)*100:+.0f}% maior '
        f'e engajamento {(eng_com-eng_sem)/max(eng_sem,1e-9)*100:+.0f}% maior'
    )

def _contexto_text_density(platform, objective, val_atual, val_novo):
    com, sem, usou_obj = _lookup(platform, 'text_density_cat', objective, val_novo, val_atual)

    if not com or not sem:
        return ''

    ctr_n = com.get('ctr_lg_mm', 0)
    ctr_a = sem.get('ctr_lg_mm', 0)

    if usou_obj:
        obj_label = OBJECTIVE_LABEL.get(objective, objective)
        prefix = f'Para campanhas de {obj_label} no {platform}'
    else:
        prefix = f'No {platform}'

    return (
        f"{prefix}, texto '{TEXT_DENSITY_MAP[int(val_novo)]}' tem "
        f"taxa de cliques {(ctr_n-ctr_a)/max(ctr_a,1e-9)*100:+.0f}% maior "
        f"que texto '{TEXT_DENSITY_MAP[int(val_atual)]}'"
    )

print('Helpers carregados.')


Helpers carregados.


In [37]:
def recomendar(row, top_n=5):
    if isinstance(row, pd.Series):
        row = row.to_dict()

    platform   = _get_platform(row)
    objective  = _get_objective(row)
    query      = pd.DataFrame([row])[feature_cols].fillna(mediana)
    score_atual = float(model.predict(query)[0])
    mudancas   = {}

    # ── DiCE com margem progressiva ───────────────────────────────────────
    for margem in [8, 5, 3]:
        try:
            cf    = exp.generate_counterfactuals(
                query,
                total_CFs=5,
                desired_range=[score_atual + margem, min(score_atual + 35, 100)],
                features_to_vary=FEATURES_RECOMENDAVEIS,
                permitted_range=permitted_range,
            )
            cf_df = cf.cf_examples_list[0].final_cfs_df

            for _, cf_row in cf_df.iterrows():
                delta = cf_row['klike_score'] - score_atual
                for feat in FEATURES_RECOMENDAVEIS:
                    val_atual = row.get(feat, mediana.get(feat, 0))
                    val_novo  = cf_row[feat]
                    # snap binarias para 0/1
                    if feat in BINARY_FEATS:
                        val_novo = round(val_novo)
                    if abs(val_novo - val_atual) > 0.05:
                        if feat not in mudancas or delta > mudancas[feat]['delta']:
                            mudancas[feat] = {
                                'delta':      round(delta, 1),
                                'valor_atual': val_atual,
                                'valor_novo':  val_novo,
                            }
            break
        except Exception:
            continue

    # ── Fallback manual se DiCE nao encontrou nada ────────────────────────
    if not mudancas:
        for feat in BINARY_FEATS:
            if row.get(feat, 1) == 0:
                new_score = float(model.predict(
                    pd.DataFrame([{**row, feat: 1}])[feature_cols].fillna(mediana)
                )[0])
                delta = round(new_score - score_atual, 1)
                if delta > 0:
                    mudancas[feat] = {'delta': delta, 'valor_atual': 0, 'valor_novo': 1}

        den_atual = int(row.get('text_density_cat', 0))
        for v in range(den_atual - 1, -1, -1):
            new_score = float(model.predict(
                pd.DataFrame([{**row, 'text_density_cat': v}])[feature_cols].fillna(mediana)
            )[0])
            delta = round(new_score - score_atual, 1)
            if delta > 0:
                mudancas['text_density_cat'] = {
                    'delta': delta, 'valor_atual': den_atual, 'valor_novo': v
                }
                break

    # ── Monta recomendacoes com contexto ─────────────────────────────────
    recs = []
    for feat, info in mudancas.items():
        if feat in BINARY_FEATS:
            contexto = _contexto_binaria(platform, feat, objective)
        elif feat == 'text_density_cat':
            contexto = _contexto_text_density(
                platform, objective, info['valor_atual'], info['valor_novo']
            )
        else:
            contexto = ''

        recs.append({
            'feature':  feat,
            'acao':     FEAT_LABEL.get(feat, feat),
            'delta':    info['delta'],
            'contexto': contexto,
        })

    recs.sort(key=lambda x: x['delta'], reverse=True)
    return score_atual, recs[:top_n]


def exibir_recomendacoes(row, top_n=5):
    if isinstance(row, pd.Series):
        row_dict = row.to_dict()
    else:
        row_dict = row

    platform  = _get_platform(row_dict)
    objective = _get_objective(row_dict)
    obj_label = OBJECTIVE_LABEL.get(objective, objective)
    _, recs   = recomendar(row_dict, top_n=top_n)

    print('=' * 65)
    print(f'  Plataforma : {platform}  |  Objetivo: {obj_label}')
    print(f'  {len(recs)} ajuste(s) identificado(s) no seu criativo')
    print('=' * 65)

    if not recs:
        print('  Nenhum ajuste necessario — criativo ja esta otimizado.')
        return

    for i, r in enumerate(recs, 1):
        print(f"\n  #{i}  {r['acao']}")
        if r['contexto']:
            print(f"       Por que: {r['contexto']}")

    print('=' * 65)


print('Engine pronto.')


Engine pronto.


## Demonstracao

Tres campanhas com perfis distintos.

In [38]:
# Campanha 1 — TikTok com score baixo (todos os atributos ausentes)
row = df_train.iloc[5]
exibir_recomendacoes(row)


100%|██████████| 1/1 [00:00<00:00,  3.09it/s]

  Plataforma : TikTok  |  Objetivo: trafego
  3 ajuste(s) identificado(s) no seu criativo

  #1  Adicionar um gancho nos primeiros 3 segundos
       Por que: Para campanhas de trafego no TikTok, videos COM gancho nos primeiros segundos tem taxa de cliques +86% maior e engajamento +19% maior

  #2  Ajustar o equilibrio entre musica e voz

  #3  Adicionar uma chamada para acao (ex: 'Saiba mais', 'Compre agora')
       Por que: Para campanhas de trafego no TikTok, videos COM chamada para acao tem taxa de cliques +12% maior e engajamento +2% maior


In [39]:
# Campanha 2 — Meta com score medio (falta hook, texto denso)
row = df_train.iloc[3]
exibir_recomendacoes(row)


100%|██████████| 1/1 [00:00<00:00,  3.30it/s]


No Counterfactuals found for the given configuration, perhaps try with different parameters... ; total time taken: 00 min 00 sec


100%|██████████| 1/1 [00:00<00:00,  3.29it/s]


No Counterfactuals found for the given configuration, perhaps try with different parameters... ; total time taken: 00 min 00 sec


100%|██████████| 1/1 [00:00<00:00,  3.03it/s]

  Plataforma : Meta  |  Objetivo: conversoes
  4 ajuste(s) identificado(s) no seu criativo

  #1  Adicionar um gancho nos primeiros 3 segundos
       Por que: Para campanhas de conversoes no Meta, videos COM gancho nos primeiros segundos tem taxa de cliques +37% maior e engajamento +18% maior

  #2  Adicionar legendas
       Por que: Para campanhas de conversoes no Meta, videos COM legendas tem taxa de cliques +20% maior e engajamento +23% maior

  #3  Ajustar o equilibrio entre musica e voz

  #4  Incluir rosto humano no video
       Por que: Para campanhas de conversoes no Meta, videos COM rosto humano tem taxa de cliques -4% maior e engajamento +30% maior


In [40]:
# Campanha 3 — LinkedIn com score medio (faltam hook e rosto)
row = df_train.iloc[8]
exibir_recomendacoes(row)


100%|██████████| 1/1 [00:00<00:00,  2.90it/s]


No Counterfactuals found for the given configuration, perhaps try with different parameters... ; total time taken: 00 min 00 sec


100%|██████████| 1/1 [00:00<00:00,  3.22it/s]


No Counterfactuals found for the given configuration, perhaps try with different parameters... ; total time taken: 00 min 00 sec


100%|██████████| 1/1 [00:00<00:00,  3.33it/s]

  Plataforma : LinkedIn  |  Objetivo: conversoes
  2 ajuste(s) identificado(s) no seu criativo

  #1  Adicionar um gancho nos primeiros 3 segundos
       Por que: Para campanhas de conversoes no LinkedIn, videos COM gancho nos primeiros segundos tem taxa de cliques +142% maior e engajamento -40% maior

  #2  Incluir rosto humano no video
       Por que: Para campanhas de conversoes no LinkedIn, videos COM rosto humano tem taxa de cliques +15% maior e engajamento +32% maior


## Analise em Lote

Roda o engine em todo o conjunto de treino e rankeia as recomendacoes por frequencia e impacto medio.


In [ ]:
resultados = []
for _, row in df_train.iterrows():
    _, recs = recomendar(row)
    for r in recs:
        resultados.append({'feature': r['feature'], 'delta': r['delta']})

resumo = (
    pd.DataFrame(resultados)
      .groupby('feature')
      .agg(frequencia=('delta','count'),
           delta_medio=('delta','mean'),
           delta_max=('delta','max'))
      .sort_values('delta_medio', ascending=False)
      .round(2)
)

print('Ranking de impacto das recomendacoes (conjunto de treino)\n')
print(resumo.to_string())
